In [ ]:
# allows update of external libraries without need to reload package
%load_ext autoreload
%autoreload 2

# Layered data validation and processing demonstration

## Summary

Demonstrate a workflow of a layered data validation and processing of WTG data.

## Results

### Key results

- Basic structure is a medaillon architecture w/ layers raw, bronze, silver and gold.
- Core functionality is a tracking, logging and reporting mechanism of the results, currently as structured summary, but opening the possibility for a reporting system.
- Validations are configurable and versionable.

### Details

- Layers (concept):
  - Raw:
    - Formal diagnostics for a standardisation.
  - Bronze:
    - Domain-related diagnostics for outlier detection.
  - Silver:
    - Gap filling.
  - Gold:
    - Advanced stuff.

## Remarks


## Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import ergaleiothiki
from phoibe.layered.application.config import ValidationConfig
from phoibe.layered.application.context import ValidationContext
from phoibe.layered.application.factory import ValidatorFactory, RuleRegistry
from phoibe.layered.infrastructure.io import YAMLReportRepository
from phoibe.layered.core.entities import RuleExecutionResult, Status, Severity

import phoibe.layered.rules.rules_index
import phoibe.layered.rules.rules_columns

## Provision

### Layer rule configuration

In [ ]:
VARIABLE_PATTERNS = {
    "power_kw": [r"power", r"production"],
    "wind_speed": [r"wind.*speed", r"wind.*speed.*avg"],
    "wind_speed_max": [r"wind.*speed.*max"],
    "datetime": [r"datetime", r"timestamp"],
    "date": [r"date"],
    "time": [r"time"],
}
RULES = [
    {"name": "required_variable", "params": {"variable_name": "power"}},
    {"name": "temporal_attributes", "params": {}},
]
validation_config = ValidationConfig(layer_name="raw", variable_patterns=VARIABLE_PATTERNS, rules=RULES)

In [ ]:
RuleRegistry().list_rules()

### Validation

In [ ]:
data = pd.DataFrame(
    data=np.random.random(size=(5, 3)),
    columns=["TIMESTAMP", "wind_speed", "power"],
    index=pd.date_range(start="2026-02-14T14:00", freq="10min", periods=5, name="datetime"),
).reset_index()

In [ ]:
validator = ValidatorFactory().create_from_memory(config=validation_config, data=data)
report = validator.validate(file_path=".", turbine_id="WEA 01")

### Reporting

In [ ]:
report.rule_execution_results

In [ ]:
YAMLReportRepository().save(reported=[report], output_path="reports/report_raw.yaml")

In [ ]:
context = ValidationContext(
    detected_variables=validator.variable_detector.detect(data), turbine_id="WEA 02", layer_name="raw"
)

In [ ]:
def execute(df: pd.DataFrame, context: ValidationContext):
    datetime_key = context.get_column_key("datetime")
    if datetime_key is None:
        return display("Datetime variable not detected.")
    try:
        # datetime_column = pd.to_datetime(df[datetime_key], errors="coerce")
        datetime_column = df[datetime_key]
    except Exception as e:
        return display("Failed to parse timestamps", exception=e)

    # start = str(datetime_column.min().isoformat())
    # end = str(datetime_column.max().isoformat())
    # tzinfo = str(datetime_column.tzinfo)
    # has_duplicates = datetime_column.has_duplicates
    # delta_datetime_column = datetime_column.diff()
    # is_sorted = not bool((delta_datetime_column.dropna() < pd.to_timedelta(0)).any())

    # result = {
    #     "start": start,
    #     "end": end,
    #     "tzinfo": tzinfo,
    #     "has_duplicates": has_duplicates,
    #     "is_sorted": is_sorted,
    # }
    # return RuleExecutionResult(
    #     rule_name="A",
    #     status=Status.WARNING,
    #     required="",
    #     actual="",
    #     message="",
    #     points_max=20,
    #     points_achieved=10,
    #     details=result,
    # )
    return datetime_column


execute(data, context)
pd.Index(pd.to_datetime(data["index"], errors="coerce")).tzinfo